## DATA INGESTION


In [2]:
### Document Structure

from langchain_core.documents import Document

In [3]:
doc = Document(
    page_content="This is main text content I am using to create RAG",
    metadata={
        'source':'bliss_corpus.json',
        'pages':1,
        'author':'Subodh',
        'date_created':'2026-09-22'
    }
)

print(doc)

page_content='This is main text content I am using to create RAG' metadata={'source': 'bliss_corpus.json', 'pages': 1, 'author': 'Subodh', 'date_created': '2026-09-22'}


In [4]:
### create a simple txt file
import os
os.makedirs('../data/text_files', exist_ok=True)

In [ ]:
# LOADING JSON FILES

from langchain_community.document_loaders import JSONLoader
from pathlib import Path

file_path = Path("../data/json_files/bliss_corpus.json")

def extract_metadata(record: dict, metadata: dict) -> dict:
    metadata["doc_id"] = record.get("doc_id")
    metadata["source"] = record.get("metadata", {}).get("source")
    metadata["url"] = record.get("metadata", {}).get("url")
    metadata["topic_group"] = record.get("metadata", {}).get("topic_group")
    metadata["flags"] = record.get("metadata", {}).get("flags")
    return metadata

## JSON loader with jq schema and content key
loader = JSONLoader(
    file_path=str(file_path),
    jq_schema=".[]",  
    content_key='. | "Question: " + .question + "\nAnswer: " + .answer',
    is_content_key_jq_parsable=True,  
    metadata_func=extract_metadata,
)

documents = loader.load()
documents[:5]

[Document(metadata={'source': 'NIMH', 'seq_num': 1, 'doc_id': 'nimh_5-action-steps-to-help-someone-having-thoughts-of-suicide_01', 'url': 'https://www.nimh.nih.gov/health/publications/5-action-steps-to-help-someone-having-thoughts-of-suicide', 'topic_group': 'suicide_selfharm_crisis', 'flags': ['safety_sensitive']}, page_content='Question: If I\'m worried someone might be suicidal, is it okay to just ask them directly?\nAnswer: Yes. Directly asking someone "Are you thinking about suicide?" is the first of the 5 action steps. Research shows that asking people if they are suicidal does not increase suicidal behavior or thoughts, and the question can help open up a conversation.'),
 Document(metadata={'source': 'NIMH', 'seq_num': 2, 'doc_id': 'nimh_5-action-steps-to-help-someone-having-thoughts-of-suicide_02', 'url': 'https://www.nimh.nih.gov/health/publications/5-action-steps-to-help-someone-having-thoughts-of-suicide', 'topic_group': 'suicide_selfharm_crisis', 'flags': ['safety_sensitiv

### Creating Data Chunks

In [12]:
### Not Performing Chunking on Json Data because:
# each document in json dataset represents a self-contained Question & Answer pair that is already short and focused
# In RAG pipelines, chunking is designed to break down large documents (like PDFs or long articles) into 
# smaller, semantically coherent pieces so that search queries match relevant paragraphs instead of huge walls of text.

### Embeddings and VectorStore DB

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

In [10]:
## BAAI/bge-small-en-v1.5 for Json dataset
from typing import List
import numpy as np
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer

class EmbeddedDocument:
    """"Pairs an Embedding Vector with it's source Document."""
    embedding: np.ndarray
    text: str
    metadata: Dict[str, Any]

class EmbeddingManager:
    """Handles document embeddings generation using SentenceTransformer and manages storage in Vector Database."""

    def __init__(self, model_name: str = "BAAI/bge-small-en-v1.5"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(
                f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}"
            )
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded.")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Embeddings generated. Shape: {embeddings.shape}")
        return embeddings


    def embed_documents(self, documents: List[Document]) -> List[EmbeddedDocument]:
        """
        Extracts page_content from LangChain Document objects and generates embeddings.
        One Vector per Document, preserving the original text and metadata.
        Args:
            documents: List of LangChain Document objects.
        Returns:
            List of EmbbeddedDocument, each holding the embedding vector, text, and metadata from the original Document.
        """
        if not documents:
            raise ValueError("No documents provided for embedding.")

        texts = [doc.page_content for doc in documents]
        embeddings = self.generate_embeddings(texts)
        return[
            EmbeddedDocument(
                embedding=embeddings[i],
                text=documents[i].page_content,
                metadata=documents[i].metadata
            )
            for i in range(len(documents))
        ]

## initialize the embedding manager for JSON dataset
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: BAAI/bge-small-en-v1.5...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2243.83it/s]


Model loaded successfully. Embedding dimension: 384


### VectorStrore

In [11]:
## Vector Store for JSON dataset
class VectorStore:
    """Manages document embeddings in a ChromaDB Vector Store"""

    def __init__(self,collection_name: str = 'json_qa_documents', persist_directory: str = '../data/vector_store'):
        """
        Initialize the vector store
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()


    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "JSON Q&A document embeddings for RAG",
                    "hnsw:space": "cosine"
                }
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def _flatten_list(self, metadata: Dict[str, Any]) -> Dict[str, Any]:
        """
        Flatten nested lists in metadata for ChromaDB compatibility.
        ChromaDB only accepts str/int/float/bool metadata values.
        JSON docs carry list fields ('flags': ['safety_sensitive', ...]),
        so flatten any list into a comma-separated string. 
        """
        clean = {}
        for k, v in metadata.items():
            if isinstance(v, list):
                clean[k] = ", ".join(str(x) for x in v)
            elif v is None:
                clean[k] = ""
            else:
                clean[k] = v
        return clean

    def add_documents(self, documents: List[Document], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = doc.metadata.get("doc_id") or f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = self._flatten_list(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: json_qa_documents
Existing documents in collection: 0


In [12]:
### Convert Text to embeddings and store in Vector Store
json_texts = [doc.page_content for doc in documents]

### Generate embeddings for the text documents
embeddings = embedding_manager.generate_embeddings(json_texts)

### Add documents and embeddings to the vector store
vectorstore.add_documents(documents, embeddings)

Generating embeddings for 2964 texts...


Batches: 100%|██████████| 93/93 [03:35<00:00,  2.32s/it]


Embeddings generated. Shape: (2964, 384)
Adding 2964 documents to vector store...
Successfully added 2964 documents to vector store
Total documents in collection: 2964


### Retriever Pipeline From VectorStore

In [21]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        '''
        Initialize the Retriever

        Args:
            vector_store: Vector Store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        '''

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 3, score_threshold: float = 0.3, where: Dict[str, Any] = None) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            where: Additional filtering criteria for the query like the Flags(suicide, safety_sensitive, etc.)
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            query_kwargs = {
                "query_embeddings": [query_embedding.tolist()],
                "n_results": top_k
            }
            
            if where:
                query_kwargs["where"] = where

            results = self.vector_store.collection.query(**query_kwargs)

            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vectorstore,embedding_manager)
rag_retriever

In [22]:
rag_retriever.retrieve(query="How should I keep someone safe from suicide?",)

Retrieving documents for query: 'How should I keep someone safe from suicide?'
Top K: 3, Score threshold: 0.3
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 42.00it/s]

Embeddings generated. Shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'nimh_5-action-steps-to-help-someone-having-thoughts-of-suicide_03',
  'content': "Question: How can I help keep someone safe if they're having suicidal thoughts?\nAnswer: You can help by reducing their access to highly lethal items or places. Ask the person if they have a specific plan, and work to make any lethal means less available or less dangerous, since this can help keep them safe when suicidal thoughts arise.",
  'metadata': {'content_length': 337,
   'url': 'https://www.nimh.nih.gov/health/publications/5-action-steps-to-help-someone-having-thoughts-of-suicide',
   'doc_id': 'nimh_5-action-steps-to-help-someone-having-thoughts-of-suicide_03',
   'doc_index': 2,
   'source': 'NIMH',
   'seq_num': 3,
   'topic_group': 'suicide_selfharm_crisis',
   'flags': 'safety_sensitive'},
  'similarity_score': 0.8576216697692871,
  'distance': 0.1423783302307129,
  'rank': 1},
 {'id': 'helios_59',
  'content': "Question: How do I stop suicidal thoughts?\nAnswer: Keep in mind that th

### Integration VectorDB Context pipeline with LLM output

In [23]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

## Initialize the Groq LLM (set your Groq_API_Key in .env)
groq_api_key = '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'
llm = ChatGroq(groq_api_key=groq_api_key, model_name='qwen/qwen3.8-27b', temperature=0.1,max_tokens=1024)

##simple RAG funtion: retrieve context + generate responses
def rag_simple(query,retriever,llm,top_k=3):

    ##retrieve the context
    results = retriever.retrieve(query,top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."

    ## generate the answer using Groq LLM
    prompt = f"""Use the following context to answer the question concisely.
                Context:
                {context}

                Question: {query}

                Answer:"""

    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content


In [24]:
answer = rag_simple("What is Mental Health?", rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is Mental Health?'
Top K: 3, Score threshold: 0.3
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 40.47it/s]

Embeddings generated. Shape: (1, 384)
Retrieved 3 documents (after filtering)


Mental health is our mental well-being, which includes our beliefs, thoughts, feelings, and behaviors. It encompasses our emotions, ability to solve problems and overcome difficulties, social connections, and understanding of the world. Unlike mental illness, which is a specific biological brain disorder, mental health is a continuum that everyone possesses, ranging from good health to poor health or illness.
